# 01 · EL image preprocessing

Exploratory preprocessing pipeline for electroluminescence (EL) images: resolution check, corrupted-file removal, border removal, CLAHE contrast enhancement, letterbox resize for the detector (640×640), resize for the classifier (224×224) and normalisation.

| | |
|---|---|
| **Input** | A folder of EL images exported in YOLO format (`DATA_DIR`) |
| **Output** | `DATA_DIR/processed/detection/` and `DATA_DIR/processed/transformer/` |
| **Runs on** | CPU (local Jupyter or Colab) |

> **Notes**
> - The *Remove Corrupted Images* cell **deletes files** from `DATA_DIR`. Run it on a copy of your data.
> - Uses `imghdr`, which exists in Python ≤ 3.12.
> - No images, labels or outputs are stored in this repository (see `DATA_NOTICE.md`).

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
DATA_DIR = "data/el_images"  # folder with the dataset exported in YOLO format
print("Dataset path:", DATA_DIR)

In [ ]:
from glob import glob

image_paths = glob(os.path.join(DATA_DIR, "**/*.jpg"), recursive=True)

print("Total images found:", len(image_paths))

Check Resolution 

In [ ]:
import cv2

resolutions = []

for img_path in image_paths:
    img = cv2.imread(img_path)
    if img is not None:
        h, w = img.shape[:2]
        resolutions.append((w, h))

print("Smallest resolution:", min(resolutions))
print("Largest resolution:", max(resolutions))

Remove Corrupted Images

In [ ]:
import imghdr

valid_images = []

for img_path in image_paths:
    if imghdr.what(img_path) is None:
        print("Removing corrupted:", img_path)
        os.remove(img_path)
    else:
        valid_images.append(img_path)

print("Valid images:", len(valid_images))

Remove Borders

In [ ]:
def remove_borders(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    coords = np.column_stack(np.where(gray > 5))
    
    if coords.size == 0:
        return image
    
    y_min, x_min = coords.min(axis=0)
    y_max, x_max = coords.max(axis=0)
    
    return image[y_min:y_max, x_min:x_max]

CLAHE

In [ ]:
def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)

    merged = cv2.merge((l_clahe, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

Resizing for detection model

In [ ]:
def letterbox(image, target_size=(640, 640)):
    h, w = image.shape[:2]
    target_w, target_h = target_size

    scale = min(target_w / w, target_h / h)
    new_w, new_h = int(w * scale), int(h * scale)

    resized = cv2.resize(image, (new_w, new_h))

    canvas = np.full((target_h, target_w, 3), 114, dtype=np.uint8)

    x_offset = (target_w - new_w) // 2
    y_offset = (target_h - new_h) // 2

    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

    return canvas

Resizing for transformer model

In [ ]:
def resize_image(image, size):
    return cv2.resize(image, size)

Normalization

In [ ]:
def normalize(image):
    return image / 255.0

In [ ]:
def show_image(title, img):
    plt.figure(figsize=(4,4))
    plt.title(title)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

Full Pipeline

In [ ]:
# Paths
processed_base = os.path.join(DATA_DIR, "processed")
detection_dir = os.path.join(processed_base, "detection")
transformer_dir = os.path.join(processed_base, "transformer")

os.makedirs(detection_dir, exist_ok=True)
os.makedirs(transformer_dir, exist_ok=True)

# Sizes
DETECTION_SIZE = (640, 640)
TRANSFORMER_SIZE = (224, 224)

# Visualization settings
VISUALIZE = True
MAX_SHOW = 3

for i, img_path in enumerate(valid_images):
    img = cv2.imread(img_path)

    if img is None:
        continue

    filename = os.path.basename(img_path)

    # ========================
    # Original
    # ========================
    if VISUALIZE and i < MAX_SHOW:
        show_image("Original", img)

    # ========================
    # Remove Borders
    # ========================
    img = remove_borders(img)

    if VISUALIZE and i < MAX_SHOW:
        show_image("After Border Removal", img)

    # ========================
    # CLAHE
    # ========================
    img = apply_clahe(img)

    if VISUALIZE and i < MAX_SHOW:
        show_image("After CLAHE", img)

    # ========================
    # YOLO Resize
    # ========================
    img_det = letterbox(img, DETECTION_SIZE)
    img_det = normalize(img_det)
    save_det = (img_det * 255).astype("uint8")

    if VISUALIZE and i < MAX_SHOW:
        show_image("Detection (Letterbox 640x640)", save_det)

    cv2.imwrite(os.path.join(detection_dir, filename), save_det)

    # ========================
    # Transformer Resize
    # ========================
    img_cls = resize_image(img, TRANSFORMER_SIZE)
    img_cls = normalize(img_cls)
    save_cls = (img_cls * 255).astype("uint8")

    if VISUALIZE and i < MAX_SHOW:
        show_image("Transformer (800x800)", save_cls)

    cv2.imwrite(os.path.join(transformer_dir, filename), save_cls)

print("Full preprocessing pipeline completed!")